# PicoRV32 + CIM SoC — PYNQ 板上验证

**Pure-PL 设计**: PicoRV32 RISC-V 软核从 BRAM 取指执行，通过 AXI-Lite 驱动 CIM IP 完成 MNIST 推理，结果通过 UART 输出。

## 验证目标

1. 加载 `cim_rv32_soc.bit`（无需 hwh）
2. 通过 USB-TTL 串口读取 PicoRV32 的 UART 输出
3. 解析推理结果，与 Golden Model 对比
4. 证明 PicoRV32 RISC-V 可以替代 ARM PS 控制 CIM 加速器

## 硬件连接

```
PYNQ-Z2 PMODA            USB-TTL
  Pin 1 (Y18, UART TX) --> RXD
  Pin 5 (GND)          --> GND
                            USB --> PYNQ USB or PC
```

串口参数: **115200 baud, 8N1, 无流控**

## 上传文件清单

| 文件 | 说明 |
|------|------|
| `cim_rv32_soc.bit` | PicoRV32 版 bitstream (checkpoint2) |
| `golden_results.txt` | 宿主机 golden model 输出 (可选) |
| `small_mlp_data/` | 模型数据目录 (可选, 用于 PS 版交叉验证) |
| `cim_soc.bit` + `cim_soc.hwh` | PS 版 bitstream (可选, 用于交叉验证) |

## 1. 加载 Bitstream（无需 hwh）

Pure-PL 设计没有 Block Design，不会生成 `.hwh` 文件。
绕过 `pynq.Overlay()`，直接写入 Zynq 配置设备。

In [ ]:
import subprocess, os, time, glob

BIT_FILE = "cim_rv32_soc.bit"

assert os.path.exists(BIT_FILE), f"{BIT_FILE} not found! Please upload."

print(f"Loading {BIT_FILE} ({os.path.getsize(BIT_FILE)/1024:.0f} KB)...")

# Pure-PL: no hwh, cannot use pynq.Overlay()
# Write directly to Zynq configuration device
try:
    subprocess.run(
        ["sudo", "bash", "-c", f"cat {BIT_FILE} > /dev/xdevcfg"],
        check=True, timeout=30
    )
    print("Bitstream loaded!")
except Exception as e:
    print(f"Method 1 failed: {e}")
    try:
        subprocess.run(["fpgautil", "-b", BIT_FILE], check=True, timeout=30)
        print("Bitstream loaded! (fpgautil)")
    except Exception as e2:
        print(f"Method 2 failed: {e2}")
        print("Manual: sudo bash -c 'cat cim_rv32_soc.bit > /dev/xdevcfg'")

print()
print("PicoRV32 is now executing firmware!")
print("  LED0 (R14): CIM done indicator")
print("  LED1 (P14): heartbeat (~1Hz blink = CPU running)")

## 2. 读取 UART 输出

PicoRV32 的推理结果通过 UART TX (PMODA Pin 1, Y18) 输出。
USB-TTL 适配器插入后通常识别为 `/dev/ttyUSB0` 或 `/dev/ttyUSB1`。

In [ ]:
# Detect serial ports
serial_ports = glob.glob("/dev/ttyUSB*") + glob.glob("/dev/ttyACM*")
print("Available serial ports:")
if serial_ports:
    for p in sorted(serial_ports):
        print(f"  {p}")
else:
    print("  No USB serial device detected!")
    print("  Check: USB-TTL adapter connected? Try: dmesg | tail")

In [ ]:
import serial

# ========== CONFIG ==========
SERIAL_PORT = "/dev/ttyUSB0"   # Modify based on detection above
BAUD_RATE = 115200
TIMEOUT_SEC = 60
# =============================

print(f"Opening {SERIAL_PORT} @ {BAUD_RATE} baud...")
print("Tip: press BTN0 (reset) to restart PicoRV32 if no output\n")

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    ser.reset_input_buffer()

    uart_output = ""
    start_time = time.time()
    done = False

    while time.time() - start_time < TIMEOUT_SEC:
        raw = ser.readline()
        if raw:
            line = raw.decode(errors="ignore").rstrip("\r\n")
            if line:
                print(line)
                uart_output += line + "\n"
            if "=== Done ===" in line:
                done = True
                break

    ser.close()
    elapsed = time.time() - start_time

    if done:
        print(f"\nInference complete! Elapsed: {elapsed:.1f}s")
    else:
        print(f"\nTimeout ({TIMEOUT_SEC}s)! Received {len(uart_output)} chars")
        if not uart_output:
            print("No data received. Check:")
            print("  1. USB-TTL RXD connected to PMODA Pin 1 (Y18)")
            print("  2. GND connected")
            print(f"  3. Correct port? (current: {SERIAL_PORT})")
            print("  4. Press BTN0 to reset")

except serial.SerialException as e:
    print(f"Serial error: {e}")
    print(f"Confirm {SERIAL_PORT} exists and is not in use")
    uart_output = ""

## 3. 解析推理结果

In [ ]:
import numpy as np

def parse_uart_output(text):
    result = {
        "predicted": None, "expected": None, "correct": None,
        "logits": [], "fc1_ok": False, "fc2_ok": False, "raw": text
    }
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("Predicted:"):
            try: result["predicted"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif line.startswith("Expected:"):
            try: result["expected"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif "CORRECT" in line:
            result["correct"] = True
        elif "WRONG" in line:
            result["correct"] = False
        elif line.startswith("Logits:"):
            nums = line.replace("Logits:", "").strip().split()
            result["logits"] = [int(x) for x in nums if x.lstrip("-").isdigit()]
        elif "FC1: done" in line:
            result["fc1_ok"] = True
        elif "FC2: computing" in line:
            result["fc2_ok"] = True
    return result

if uart_output:
    result = parse_uart_output(uart_output)
    print("=" * 50)
    print(" PicoRV32 Inference Result")
    print("=" * 50)
    print(f"  FC1 (784->16, ReLU): {'OK' if result['fc1_ok'] else 'N/A'}")
    print(f"  FC2 (16->10):        {'OK' if result['fc2_ok'] else 'N/A'}")
    print(f"  Predicted: {result['predicted']}")
    print(f"  Expected:  {result['expected']}")
    status = 'CORRECT' if result['correct'] else ('WRONG' if result['correct'] is not None else 'UNKNOWN')
    print(f"  Result:    {status}")
    if result["logits"]:
        print(f"  Logits:    {result['logits']}")
        print(f"  ArgMax:    {np.argmax(result['logits'])}")
    print("=" * 50)
else:
    result = None
    print("No UART output to parse")

## 4. 与 Golden Model 对比

如果上传了 `golden_results.txt`（宿主机 `prepare_picorv32_env.ipynb` 生成），
可以验证硬件结果是否与 Python golden model bit-exact 一致。

In [ ]:
golden_file = "golden_results.txt"

if result and result["predicted"] is not None and os.path.exists(golden_file):
    golden = {}
    with open(golden_file) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 3:
                golden[int(parts[0])] = {"label": int(parts[1]), "pred": int(parts[2])}
    print(f"Golden results: {len(golden)} entries")

    exp = result["expected"]
    pred = result["predicted"]
    matched_idx = None
    for idx, g in golden.items():
        if g["label"] == exp:
            matched_idx = idx
            break

    if matched_idx is not None:
        g = golden[matched_idx]
        print(f"\nMatched golden record: image #{matched_idx}")
        print(f"  Golden:   label={g['label']}, pred={g['pred']}")
        print(f"  Hardware: label={exp}, pred={pred}")
        if pred == g["pred"]:
            print(f"\n  >>> MATCH - Hardware and Golden Model bit-exact! <<<")
        else:
            print(f"\n  >>> MISMATCH - HW pred={pred}, Golden pred={g['pred']} <<<")
    else:
        print(f"No golden record found for label={exp}")
elif result and result["predicted"] is not None:
    print("golden_results.txt not found, skipping comparison")
else:
    print("No inference result to compare")

## 5. 重新运行测试

按下 **BTN0** (复位按钮) 可重启 PicoRV32，固件会重新执行推理。
或者重新加载 bitstream（会复位整个 PL）:

In [ ]:
def reload_and_capture(bit_file=BIT_FILE, port=SERIAL_PORT,
                       baud=BAUD_RATE, timeout=TIMEOUT_SEC):
    print(f"Reloading {bit_file}...")
    subprocess.run(
        ["sudo", "bash", "-c", f"cat {bit_file} > /dev/xdevcfg"],
        check=True, timeout=30
    )
    time.sleep(0.5)  # Wait for MMCM lock + reset release

    ser = serial.Serial(port, baud, timeout=1)
    ser.reset_input_buffer()

    output = ""
    t0 = time.time()
    while time.time() - t0 < timeout:
        raw = ser.readline()
        if raw:
            line = raw.decode(errors="ignore").rstrip("\r\n")
            if line:
                print(line)
                output += line + "\n"
            if "=== Done ===" in line:
                break
    ser.close()
    return parse_uart_output(output)

# Uncomment to re-run:
# result2 = reload_and_capture()

## 6. 多图测试说明

当前 Pure-PL 设计中，固件（包含测试图片）在 Vivado 综合时 bake 进 BRAM。
**每个 bitstream 只能测试一张图片。**

### 多图验证的两种方式

**方式 A: VCS 仿真（推荐）**

宿主机编译 20 个 `firmware.hex` → VCS 虚拟机批量仿真:

```bash
bash hw/scripts/run_tb_rv32_batch.sh
```

**方式 B: PS 版 PYNQ 交叉验证**

使用 checkpoint1 的 `cim_soc.bit` (PS 版)，通过 Python MMIO 驱动同一个 CIM IP，
跑 20 张图。两个版本使用完全相同的权重和 CIM 硬件，结果应 bit-exact 一致。

## 7. [可选] PS 版 CIM 交叉验证 (20 张图)

需要上传: `cim_soc.bit` + `cim_soc.hwh` (从 `bitstream&hwh/checkpoint1/`) 和 `small_mlp_data/` 目录。

In [ ]:
# ====== PS version cross-validation: 20 images ======
# Auto-skip if checkpoint1 files not present

PS_BIT = "cim_soc.bit"
PS_HWH = "cim_soc.hwh"
DATA_DIR = "small_mlp_data"

if not all(os.path.exists(f) for f in [PS_BIT, PS_HWH]):
    print(f"Skip: need {PS_BIT} + {PS_HWH} (checkpoint1)")
    print("Copy from bitstream&hwh/checkpoint1/")
    ps_results = None
elif not os.path.isdir(DATA_DIR):
    print(f"Skip: need {DATA_DIR}/ directory")
    ps_results = None
else:
    from pynq import Overlay, MMIO

    ol = Overlay(PS_BIT)
    print("PS Overlay loaded!")
    mmio = MMIO(0x40000000, 0x4000)

    # Connectivity test
    mmio.write(0x010, 784)
    assert mmio.read(0x010) == 784, "AXI R/W failed!"

    # --- Driver functions (mirror firmware.c CIM control) ---
    def soft_reset():
        mmio.write(0x000, 0x4)

    def configure(in_dim, out_dim, zp, mult, shift, relu):
        mmio.write(0x010, in_dim)
        mmio.write(0x014, out_dim)
        mmio.write(0x018, (in_dim + 15) // 16)
        mmio.write(0x01C, (out_dim + 15) // 16)
        mmio.write(0x020, mult)
        mmio.write(0x024, shift)
        mmio.write(0x028, int(zp) & 0xFFFFFFFF)
        mmio.write(0x02C, 1 if relu else 0)

    def load_weights(chunks):
        mmio.write(0x044, 0)
        mmio.write(0x04C, 0x02)
        for c in chunks:
            mmio.write(0x048, int(c) & 0xFFFFFFFF)
        mmio.write(0x04C, 0x00)

    def load_bias(bias, n):
        for i in range(n):
            mmio.write(0x2000 + 4 * i, int(bias[i]) & 0xFFFFFFFF)

    def load_input(data):
        padded = ((len(data) + 15) // 16) * 16
        for i in range(padded):
            mmio.write(0x1000 + 4 * i, int(data[i]) if i < len(data) else 0)

    def start_wait(t=5.0):
        mmio.write(0x000, 0x2)
        mmio.write(0x000, 0x1)
        t0 = time.time()
        while not (mmio.read(0x004) & 0x2):
            if time.time() - t0 > t:
                raise TimeoutError("CIM compute timeout!")
        return time.time() - t0

    def read_logits(n):
        return [np.int8(mmio.read(0x100 + 4 * i) & 0xFF) for i in range(n)]

    # --- Load model data ---
    def read_hex_u32(p):
        with open(p) as f:
            return [int(l.strip(), 16) for l in f if l.strip()]

    def read_hex_u8(p):
        with open(p) as f:
            return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

    qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
    zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
    fc1_mult, fc1_shift = qp[0], qp[1]
    fc2_mult, fc2_shift = qp[2], qp[3]
    hw_zp1 = np.int32(np.uint32(zps[0]))
    hw_zp2 = np.int32(np.uint32(zps[1]))
    fc1_wc = read_hex_u32(f"{DATA_DIR}/fc1_weight_tiles.hex")
    fc2_wc = read_hex_u32(f"{DATA_DIR}/fc2_weight_tiles.hex")
    fc1_bias = read_hex_u32(f"{DATA_DIR}/fc1_bias.hex")
    fc2_bias = read_hex_u32(f"{DATA_DIR}/fc2_bias.hex")

    print(f"FC1: {len(fc1_wc)} chunks, mult={fc1_mult}, shift={fc1_shift}, zp={hw_zp1}")
    print(f"FC2: {len(fc2_wc)} chunks, mult={fc2_mult}, shift={fc2_shift}, zp={hw_zp2}")

    # --- Inference: 20 images ---
    test_dir = f"{DATA_DIR}/test_images"
    n_imgs = len([f for f in os.listdir(test_dir) if f.endswith("_label.txt")])
    print(f"\nTesting {n_imgs} images...\n")

    ps_results = []
    for i in range(n_imgs):
        prefix = f"img_{i:04d}"
        img = np.array(read_hex_u8(f"{test_dir}/{prefix}.hex"), dtype=np.uint8)
        label = int(open(f"{test_dir}/{prefix}_label.txt").read().strip())

        # FC1: 784 -> 16, ReLU
        soft_reset()
        configure(784, 16, hw_zp1, fc1_mult, fc1_shift, relu=True)
        load_weights(fc1_wc)
        load_bias(fc1_bias, 16)
        load_input(img)
        t1 = start_wait()
        fc1_out = np.array(read_logits(16), dtype=np.int8)

        # FC2: 16 -> 10, no ReLU
        configure(16, 10, hw_zp2, fc2_mult, fc2_shift, relu=False)
        load_weights(fc2_wc)
        load_bias(fc2_bias, 10)
        load_input(fc1_out.view(np.uint8))
        t2 = start_wait()

        logits = read_logits(10)
        pred = int(np.argmax(logits))
        ok = pred == label
        ps_results.append({
            "idx": i, "label": label, "pred": pred,
            "correct": ok, "logits": list(logits),
            "time_ms": (t1 + t2) * 1000
        })
        mark = "OK" if ok else "WRONG"
        print(f"  [{i:04d}] label={label} pred={pred} {mark}  logits={list(logits)}  ({(t1+t2)*1000:.1f}ms)")

    correct = sum(1 for r in ps_results if r["correct"])
    print(f"\nPS accuracy: {correct}/{n_imgs} ({100*correct/n_imgs:.0f}%)")

## 8. 三方对比汇总

In [ ]:
# Summary: PicoRV32 (UART) vs PS (MMIO) vs Golden Model
print("=" * 60)
print(" Summary: PicoRV32 HW / PS HW / Golden Model")
print("=" * 60)
print()

# PicoRV32 result (from UART, single image)
if result and result["predicted"] is not None:
    rv_status = "CORRECT" if result["correct"] else "WRONG"
    print(f"[PicoRV32] Predicted={result['predicted']}, "
          f"Expected={result['expected']}, {rv_status}")
else:
    print("[PicoRV32] No result (UART not connected or no data)")

# PS result
try:
    if ps_results is not None:
        c = sum(1 for r in ps_results if r["correct"])
        print(f"[PS]       {c}/{len(ps_results)} correct")
    else:
        print("[PS]       Not run (need cim_soc.bit + cim_soc.hwh)")
except NameError:
    ps_results = None
    print("[PS]       Not run")

# Golden Model
golden_file = "golden_results.txt"
if os.path.exists(golden_file):
    golden = {}
    with open(golden_file) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 4:
                golden[int(parts[0])] = {
                    "label": int(parts[1]), "pred": int(parts[2]), "ok": int(parts[3])
                }
    c = sum(1 for v in golden.values() if v["ok"])
    print(f"[Golden]   {c}/{len(golden)} correct")

    # Per-image comparison if PS also ran
    if ps_results is not None:
        all_match = True
        print(f"\n{'Image':>6} {'Label':>6} {'PS':>4} {'Golden':>7} {'Match':>6}")
        print("-" * 40)
        for r in ps_results:
            i = r["idx"]
            if i in golden:
                g = golden[i]["pred"]
                m = "OK" if r["pred"] == g else "FAIL"
                if r["pred"] != g: all_match = False
            else:
                g = "?"; m = "-"
            print(f"{i:6d} {r['label']:6d} {r['pred']:4d} {str(g):>7} {m:>6}")
        print("-" * 40)
        if all_match:
            print("\n>>> PS vs Golden: ALL bit-exact match! <<<")
        else:
            print("\n>>> WARNING: MISMATCH detected! <<<")
else:
    print("[Golden]   golden_results.txt not found")

print()
print("=" * 60)
print("Verification chain:")
print("  Python Golden <-> VCS sim (PicoRV32+CIM) <-> PYNQ (PS+CIM)")
print("  Same CIM IP + weights + quantization params")
print("  bit-exact match = PicoRV32 can replace ARM PS")
print("=" * 60)

## 结论

| 验证层 | 方式 | 控制方 | 输出 |
|--------|------|--------|------|
| Python Golden Model | `prepare_picorv32_env.ipynb` 宿主机 | Python | 基准参考|
| VCS 仿真 (20 张) | `run_tb_rv32_batch.sh` VCS 虚拟机 | PicoRV32 C 固件 | UART (仿真) |
| FPGA 硬件 (1 张) | **本 notebook** Cell 2 | PicoRV32 C 固件 | UART (物理) |
| PS 交叉验证 (20 张) | 本 notebook Cell 7 | ARM PS Python | MMIO |

```
Python Golden  <-->  VCS sim (PicoRV32+CIM)  <-->  PYNQ HW (PS+CIM)
      |                      |                              |
  same weights          same CIM IP                   same CIM IP
```

如果所有层的结果 **bit-exact 一致**，则证明:

1. CIM 硬件 IP 在 RTL 仿真和 FPGA 上行为一致
2. PicoRV32 RISC-V 固件正确驱动 CIM IP
3. PicoRV32 可以替代 ARM PS 作为 CIM 加速器的控制核心